# TP 1: LDA/QDA y optimización matemática de modelos

---
Universidad de Buenos Aires

CEIA - Análisis Matemático

Miguel Augusto Azar

a2702

---

El enunciado del TP pide que la prediccion que se hace con QDA se haga mucho más rápido usando varios metodos.

(El cuaderno esta autocontenido y funciona tanto en Google Colab como en Jupyter local.

Estructura:
1. Configuración
2. Código provisto (sin modificar) y utilidades
3. Respuestas a las preguntas Q3 a Q7
4. Verificación del código provisto
5. Tensorización (puntos 1 a 7)
6. Cholesky (puntos 8 a 14)
7. Benchmark
8. Versiones utilizadas
9. Conclusiones



## 1. Configuración

In [21]:
import sys, time, tracemalloc
import numpy as np
import numpy.linalg as LA
import pandas as pd
import scipy
from numpy.random import RandomState
from scipy.linalg import cholesky, solve_triangular
from scipy.linalg.lapack import dtrtri
from scipy.stats import multivariate_normal as mvn
from sklearn.datasets import fetch_openml, load_iris, load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm   # el código provisto usa tqdm.notebook; tqdm.auto funciona igual en Colab y en consola

## 2. Código provisto (sin modificar)

In [22]:
# --- base/bayesian.py ---
class BaseBayesianClassifier:
  def __init__(self):
    pass

  def _estimate_a_priori(self, y):
    a_priori = np.bincount(y.flatten().astype(int)) / y.size
    # Q3: para que sirve bincount?
    return np.log(a_priori)

  def _fit_params(self, X, y):
    # estimate all needed parameters for given model
    raise NotImplementedError()

  def _predict_log_conditional(self, x, class_idx):
    # predict the log(P(x|G=class_idx)), the log of the conditional probability of x given the class
    # this should depend on the model used
    raise NotImplementedError()

  def fit(self, X, y, a_priori=None):
    # if it's needed, estimate a priori probabilities
    self.log_a_priori = self._estimate_a_priori(y) if a_priori is None else np.log(a_priori)

    # now that everything else is in place, estimate all needed parameters for given model
    self._fit_params(X, y)
    # Q4: por que el _fit_params va al final? no se puede mover a, por ejemplo, antes de la priori?

  def predict(self, X):
    # this is actually an individual prediction encased in a for-loop
    m_obs = X.shape[1]
    y_hat = np.empty(m_obs, dtype=int)

    for i in range(m_obs):
      y_hat[i] = self._predict_one(X[:,i].reshape(-1,1))

    # return prediction as a row vector (matching y)
    return y_hat.reshape(1,-1)

  def _predict_one(self, x):
    # calculate all log posteriori probabilities (actually, +C)
    log_posteriori = [ log_a_priori_i + self._predict_log_conditional(x, idx) for idx, log_a_priori_i
                  in enumerate(self.log_a_priori) ]

    # return the class that has maximum a posteriori probability
    return np.argmax(log_posteriori)

In [23]:
# --- base/qda.py ---
class QDA(BaseBayesianClassifier):

  def _fit_params(self, X, y):
    # estimate each covariance matrix
    self.inv_covs = [LA.inv(np.cov(X[:,y.flatten()==idx], bias=True))
                      for idx in range(len(self.log_a_priori))]
    # Q5: por que hace falta el flatten y no se puede directamente X[:,y==idx]?
    # Q6: por que se usa bias=True en vez del default bias=False?
    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]
    # Q7: que hace axis=1? por que no axis=0?

  def _predict_log_conditional(self, x, class_idx):
    # predict the log(P(x|G=class_idx)), the log of the conditional probability of x given the class
    # this should depend on the model used
    inv_cov = self.inv_covs[class_idx]
    unbiased_x =  x - self.means[class_idx]
    return 0.5*np.log(LA.det(inv_cov)) -0.5 * unbiased_x.T @ inv_cov @ unbiased_x


class TensorizedQDA(QDA):

    def _fit_params(self, X, y):
        # ask plain QDA to fit params
        super()._fit_params(X,y)

        # stack onto new dimension
        self.tensor_inv_cov = np.stack(self.inv_covs)
        self.tensor_means = np.stack(self.means)

    def _predict_log_conditionals(self,x):
        unbiased_x = x - self.tensor_means
        inner_prod = unbiased_x.transpose(0,2,1) @ self.tensor_inv_cov @ unbiased_x

        return 0.5*np.log(LA.det(self.tensor_inv_cov)) - 0.5 * inner_prod.flatten()

    def _predict_one(self, x):
        # return the class that has maximum a posteriori probability
        return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))

In [24]:
# --- base/cholesky.py ---
class QDA_Chol1(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.L_invs = [
        LA.inv(cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True))
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L_inv = self.L_invs[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = L_inv @ unbiased_x

    return np.log(L_inv.diagonal().prod()) -0.5 * (y**2).sum()


class QDA_Chol2(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.Ls = [
        cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True)
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L = self.Ls[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = solve_triangular(L, unbiased_x, lower=True)

    return -np.log(L.diagonal().prod()) -0.5 * (y**2).sum()


class QDA_Chol3(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.L_invs = [
        dtrtri(cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True), lower=1)[0]
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L_inv = self.L_invs[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = L_inv @ unbiased_x

    return np.log(L_inv.diagonal().prod()) -0.5 * (y**2).sum()

### Utilidades de datos y benchmark

Funciones copiadas de `utils/datasets.py` y `utils/bench.py`. Wine e Iris vienen con scikit-learn. Letters se descarga de OpenML al ejecutar.

In [25]:
# --- utils/datasets.py ---
def get_iris_dataset():
  data = load_iris()
  X_full = data.data
  y_full = np.array([data.target_names[y] for y in data.target.reshape(-1,1)])
  return X_full, y_full

def get_penguins_dataset():
    # get data
    df, tgt = fetch_openml(name="penguins", return_X_y=True, as_frame=True, parser='auto')

    # drop non-numeric columns
    df.drop(columns=["island","sex"], inplace=True)

    # drop rows with missing values
    mask = df.isna().sum(axis=1) == 0
    df = df[mask]
    tgt = tgt[mask]

    return df.values, tgt.to_numpy().reshape(-1,1)

def get_wine_dataset():
    # get data
    data = load_wine()
    X_full = data.data
    y_full = np.array([data.target_names[y] for y in data.target.reshape(-1,1)])
    return X_full, y_full

def get_letters_dataset():
    # get data
    letter = fetch_openml('letter', version=1, as_frame=False)
    return letter.data, letter.target.reshape(-1,1)

def label_encode(y_full):
    return LabelEncoder().fit_transform(y_full.flatten()).reshape(y_full.shape)

def split_transpose(X, y, test_size, random_state):
    # X_train, X_test, y_train, y_test but all transposed
    return [elem.T for elem in train_test_split(X, y, test_size=test_size, random_state=random_state)]

In [26]:
# --- utils/bench.py ---
RNG_SEED = 6553

class Benchmark:
    def __init__(self, X, y, n_runs=1000, warmup=100, mem_runs=100, test_sz=0.3, rng_seed=RNG_SEED, same_splits=True):
        self.X = X
        self.y = y
        self.n = n_runs
        self.warmup = warmup
        self.mem_runs = mem_runs
        self.test_sz = test_sz
        self.det = same_splits
        if self.det:
            self.rng_seed = rng_seed
        else:
            self.rng = RandomState(rng_seed)

        self.data = dict()

        print("Benching params:")
        print("Total runs:",self.warmup+self.mem_runs+self.n)
        print("Warmup runs:",self.warmup)
        print("Peak Memory usage runs:", self.mem_runs)
        print("Running time runs:", self.n)
        approx_test_sz = int(self.y.size * self.test_sz)
        print("Train size rows (approx):",self.y.size - approx_test_sz)
        print("Test size rows (approx):",approx_test_sz)
        print("Test size fraction:",self.test_sz)

    def bench(self, model_class, **kwargs):
        name = model_class.__name__
        time_data = np.empty((self.n, 3), dtype=float)  # train_time, test_time, accuracy
        mem_data = np.empty((self.mem_runs, 2), dtype=float)  # train_peak_mem, test_peak_mem
        rng = RandomState(self.rng_seed) if self.det else self.rng


        for i in range(self.warmup):
            # Instantiate model with error check for unsupported parameters
            model = model_class(**kwargs)

            # Generate current train-test split
            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )
            # Run training and prediction (timing or memory measurement not recorded)
            model.fit(X_train, y_train)
            model.predict(X_test)

        for i in tqdm(range(self.mem_runs), total=self.mem_runs, desc=f"{name} (MEM)"):

            model = model_class(**kwargs)

            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )

            tracemalloc.start()

            t1 = time.perf_counter()
            model.fit(X_train, y_train)
            t2 = time.perf_counter()

            _, train_peak = tracemalloc.get_traced_memory()
            tracemalloc.reset_peak()

            model.predict(X_test)
            t3 = time.perf_counter()
            _, test_peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()

            mem_data[i,] = (
                train_peak / (1024 * 1024),
                test_peak / (1024 * 1024)
            )

        for i in tqdm(range(self.n), total=self.n, desc=f"{name} (TIME)"):
            model = model_class(**kwargs)

            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )

            t1 = time.perf_counter()
            model.fit(X_train, y_train)
            t2 = time.perf_counter()
            preds = model.predict(X_test)
            t3 = time.perf_counter()

            time_data[i,] = (
                (t2 - t1) * 1000,
                (t3 - t2) * 1000,
                (y_test.flatten() == preds.flatten()).mean()
            )

        self.data[name] = (time_data, mem_data)

    def summary(self, baseline=None):
        aux = []
        for name, (time_data, mem_data) in self.data.items():
            result = {
                'model': name,
                'train_median_ms': np.median(time_data[:, 0]),
                'train_std_ms': time_data[:, 0].std(),
                'test_median_ms': np.median(time_data[:, 1]),
                'test_std_ms': time_data[:, 1].std(),
                'mean_accuracy': time_data[:, 2].mean(),
                'train_mem_median_mb': np.median(mem_data[:, 0]),
                'train_mem_std_mb': mem_data[:, 0].std(),
                'test_mem_median_mb': np.median(mem_data[:, 1]),
                'test_mem_std_mb': mem_data[:, 1].std()
            }
            aux.append(result)
        df = pd.DataFrame(aux).set_index('model')

        if baseline is not None and baseline in self.data:
            df['train_speedup'] = df.loc[baseline, 'train_median_ms'] / df['train_median_ms']
            df['test_speedup'] = df.loc[baseline, 'test_median_ms'] / df['test_median_ms']
            df['train_mem_reduction'] = df.loc[baseline, 'train_mem_median_mb'] / df['train_mem_median_mb']
            df['test_mem_reduction'] = df.loc[baseline, 'test_mem_median_mb'] / df['test_mem_median_mb']
        return df

## 3. Respuestas a las preguntas Q3 a Q7

**Q3. ¿Para qué sirve `bincount`?**
Cuenta cuántas veces aparece cada número. Si las clases son 0, 1, 2, indica cuantos ejemplos hay de cada una. Dividido por el total, da la frecuencia de cada clase, es decir, la probabilidad *a priori*. Por eso las clases tienen que ser enteros de 0 a $k-1$.

**Q4. ¿Por que `_fit_params` va al final? ¿Se puede mover antes?**
No, porque necesita dos cosas que ya deben estar listas: las clases como numeros de 0 a $k-1$, y `self.log_a_priori`, de cuya longitud se saca cuántas clases hay (`range(len(self.log_a_priori))`). Primero se averigua cuantas clases hay y cuánto pesa cada una, recién despues se calcula, clase por clase, su media y su covarianza.

**Q5. ¿Porque hace falta `flatten` en `X[:, y.flatten()==idx]`?**
Porque `y` tiene forma $(1, n)$, es una *fila*. Entonces `y == idx` es una tabla de una fila y NumPy la interpreta como un indice extra y da error (se comprueba abajo). Con `flatten()` queda una lista simple de $n$ verdaderos/falsos que dice "esta columna sí, esta no".

**Q6. ¿Por que `bias=True`?**
Porque así la covarianza se divide por $n$ y no por $n-1$. El modelo se define con el estimador de máxima verosimilitud, que divide por $n$. La versión con $n-1$ es una corrección para muestras chicas; con muchos datos casi no hay diferencia.

**Q7. ¿Que hace `axis=1`? ¿Porque no `axis=0`?**
Los datos están con las variables en filas y las observaciones en columnas. `axis=1` promedia a lo largo de las columnas, o sea sobre todas las observaciones, y da una media por variable. `axis=0` promediaría las variables de una misma clase entre sí , lo cual no tiene sentido.

In [27]:
# Comprobación de Q5
X_demo = np.random.randn(3, 6); y_demo = np.array([[0, 1, 0, 1, 0, 1]])
try:
    X_demo[:, y_demo == 0]
except IndexError as e:
    print("Sin flatten:", type(e).__name__, "-", e)
print("Con flatten:", X_demo[:, y_demo.flatten() == 0].shape)

Sin flatten: IndexError - too many indices for array: array is 2-dimensional, but 3 were indexed
Con flatten: (3, 3)


## 4. Verificación del código provisto

Para comprobar que el código provisto está bien, se lo compara con una referencia independiente de `scipy` (`multivariate_normal`). Se comparan tanto la clase predicha como el puntaje de cada clase.

In [28]:
X_wine, y_wine = get_wine_dataset()
y_wine_enc = label_encode(y_wine)
Xtr, Xte, ytr, yte = split_transpose(X_wine, y_wine_enc, test_size=0.3, random_state=1)
k_wine, p_wine = len(np.unique(y_wine_enc)), Xtr.shape[0]
print("Xtr", Xtr.shape, "ytr", ytr.shape, "Xte", Xte.shape)

# referencia: log f_j(x) sin la constante, y predicción final
ref_cond = np.zeros((k_wine, Xte.shape[1]))
for j in range(k_wine):
    Xj = Xtr[:, ytr.ravel() == j]
    ref_cond[j] = mvn(Xj.mean(1), np.cov(Xj, bias=True)).logpdf(Xte.T) + 0.5 * p_wine * np.log(2 * np.pi)
log_prior = np.log(np.bincount(ytr.ravel()) / ytr.size)
pred_ref = (log_prior[:, None] + ref_cond).argmax(0)

q = QDA(); q.fit(Xtr, ytr)
cond_q = np.array([[q._predict_log_conditional(Xte[:, [i]], j).item() for i in range(Xte.shape[1])] for j in range(k_wine)])
print("QDA: log-condicional igual a la referencia:", np.allclose(cond_q, ref_cond))
print("QDA: predicciones iguales a la referencia:", (q.predict(Xte).ravel() == pred_ref).all())

Xtr (13, 124) ytr (1, 124) Xte (13, 54)
QDA: log-condicional igual a la referencia: True
QDA: predicciones iguales a la referencia: True


## 5. Tensorización

### Punto 1. `QDA` frente a `TensorizedQDA`

**¿Sobre qué paraleliza `TensorizedQDA`?** Solo sobre las $k$ clases. Para predecir muchas observaciones sigue recorriéndolas de a una (el `for` de `predict`), pero ya no recorre las clases de a una.

Es decir, que para una clase nueva, `QDA` calcula un puntaje para por ejemplo la clase1, después otro para la clase2, y así con cada clase. `TensorizedQDA` apila los datos de todas las clases en un solo bloque y calcula todos los puntajes de golpe.

**Paso a paso** (para una observación `x` de forma $(p,1)$):

| Paso | Qué hace | Forma |
|---|---|---|
| 1 | `tensor_means`: las medias de todas las clases, apiladas | $(k,p,1)$ |
| 2 | `tensor_inv_cov`: las inversas de las covarianzas, apiladas | $(k,p,p)$ |
| 3 | `x - tensor_means`: a `x` se le resta la media de cada clase | $(k,p,1)$ |
| 4 | `transpose(0,2,1) @ tensor_inv_cov @ ...`: mide qué tan lejos está `x` del centro de cada clase, según la forma de su nube de puntos | $(k,1,1)$ |
| 5 | `flatten()`: un número por clase | $(k,)$ |

A ese número se le suma el término del determinante y la probabilidad a priori, y `argmax` elige la clase con mayor puntaje. Es la misma cuenta que hace `QDA`, solo que con todas las clases a la vez. (`0.5*log(det(inv_cov))` equivale a $-\tfrac12\log|\Sigma|$, porque el determinante de la inversa es 1 sobre el determinante.)

### Punto 2. Optimización

**Puntos 3 y 4. `FasterQDA` y la matriz $n \times n$.** `FasterQDA` elimina también el `for` de las observaciones, o sea que procesa todas juntas. El problema es que, al hacerlo, la cuenta produce por cada clase una matriz de $n \times n$: compara cada observación con todas las demás, aunque  solo sirve la diagonal (cada observación consigo misma). Esa matriz aparece en la línea marcada `<- matriz n x n` del código, y su tamaño se imprime en la celda siguiente.

In [29]:
class FasterQDA(TensorizedQDA):
    """Predice todas las observaciones a la vez, construyendo la matriz (k, n, n)."""

    def _predict_log_conditionals_batch(self, X):
        unbiased_x = X[np.newaxis, :, :] - self.tensor_means                      # (k, p, n)
        full = unbiased_x.transpose(0, 2, 1) @ self.tensor_inv_cov @ unbiased_x   # (k, n, n)  <- matriz n x n
        quad = np.diagonal(full, axis1=1, axis2=2)                                # (k, n)
        return 0.5 * np.log(LA.det(self.tensor_inv_cov))[:, None] - 0.5 * quad

    def predict(self, X):
        scores = self.log_a_priori[:, None] + self._predict_log_conditionals_batch(X)  # (k, n)
        return np.argmax(scores, axis=0).reshape(1, -1)

In [30]:
f = FasterQDA(); f.fit(Xtr, ytr)
U = Xte[np.newaxis] - f.tensor_means
full = U.transpose(0, 2, 1) @ f.tensor_inv_cov @ U
print("Forma de la matriz intermedia (k, n, n):", full.shape)
print("Memoria de esa matriz con Letters (k=26, n=4000):", round(8 * 26 * 4000**2 / 2**30, 2), "GiB")

Forma de la matriz intermedia (k, n, n): (3, 54, 54)
Memoria de esa matriz con Letters (k=26, n=4000): 3.1 GiB


**Punto 5. Demostración de que $\mathrm{diag}(A B) = \texttt{np.sum}(A \odot B^T,\ \texttt{axis}=1)$.**

El elemento $i$ de la diagonal de $AB$ es "fila $i$ de $A$ por columna $i$ de $B$",ees decir que se multiplican termino a termino y se suma. Eso es justo lo que hace $A \odot B^T$ (multiplicar termino a término) seguido de sumar cada fila. No hace falta calcular el resto de la matriz.

En fórmulas, con $A$ de $n \times p$ y $B$ de $p \times n$:

$$(AB)_{ii} = \sum_{j=1}^{p} A_{ij}\,B_{ji} = \sum_{j=1}^{p} A_{ij}\,(B^T)_{ij} = \sum_{j=1}^{p} (A \odot B^T)_{ij}$$

que es la suma de la fila $i$ de $A \odot B^T$ (`axis=1`). $\blacksquare$

Forma equivalente: `np.sum(A.T * B, axis=0)`.

La ventaja es que solo se usan matrices de $n \times p$ y no de $n \times n$. Con $p=16$ y $n=4000$, la diferencia es muy grande.

In [31]:
A = np.random.randn(7, 4); B = np.random.randn(4, 7)
print(np.allclose(np.diag(A @ B), np.sum(A * B.T, axis=1)),
      np.allclose(np.diag(A @ B), np.sum(A.T * B, axis=0)))

True True


**Punto 6. `EfficientQDA`.** Se usa la propiedad anterior con $A = U^T\Sigma^{-1}$ (de forma $(n,p)$ por clase) y $B = U$ (de forma $(p,n)$), donde $U$ son las observaciones ya centradas. Así se obtiene el puntaje de cada observación sin armar nunca la matriz $n \times n$.

In [32]:
class EfficientQDA(TensorizedQDA):
    """Igual que FasterQDA pero sin materializar la matriz n x n."""

    def _predict_log_conditionals_batch(self, X):
        unbiased_x = X[np.newaxis, :, :] - self.tensor_means    # (k, p, n)
        Ut = unbiased_x.transpose(0, 2, 1)                      # (k, n, p)
        quad = np.sum((Ut @ self.tensor_inv_cov) * Ut, axis=2)  # (k, n)
        return 0.5 * np.log(LA.det(self.tensor_inv_cov))[:, None] - 0.5 * quad

    def predict(self, X):
        scores = self.log_a_priori[:, None] + self._predict_log_conditionals_batch(X)
        return np.argmax(scores, axis=0).reshape(1, -1)

## 6. Cholesky

Cholesky escribe la covarianza como $\Sigma = LL^T$, con $L$ triangular. Es casi como sacarle una "raíz cuadrada" a la matriz. Las matrices triangulares son mucho más fáciles de invertir y de usar.

### Punto 8. La inversa en términos de $L$
Si $A = LL^T$, entonces $A^{-1} = (L^{-1})^T L^{-1}$. La cuenta de QDA queda así:

$$(x-\mu)^T\Sigma^{-1}(x-\mu) = \big\|L^{-1}(x-\mu)\big\|^2$$

Es decir que se calcula $z = L^{-1}(x-\mu)$, se eleva al cuadrado cada componente y se suma. Además, $\log|\Sigma| = 2\sum_i \log L_{ii}$ con lo cual el determinante sale directo de la diagonal de $L$.

### Punto 9. `QDA_Chol1` frente a `QDA`
`QDA` invierte la covarianza completa y, en cada prediccion, calcula un determinante. `QDA_Chol1` hace esto:
1. Calcula $L$ (Cholesky) y guarda su inversa $L^{-1}$.
2. Al predecir, centra $x$ y calcula $z = L^{-1}(x-\mu)$.
3. Devuelve $\log\left(\prod_i (L^{-1})_{ii}\right) - \tfrac12 \sum z^2$.


El primer término es el log-determinante, porque el producto de la diagonal de $L^{-1}$ vale $|\Sigma|^{-1/2}$.

### Punto 10. `QDA_Chol1`, `QDA_Chol2` y `QDA_Chol3`
- `Chol1`: guarda $L^{-1}$ calculada con `LA.inv`, una inversa general que no aprovecha que $L$ es triangular.
- `Chol2`: no invierte nada. Guarda $L$ y en cada predicción resuelve el sistema $Lz = x-\mu$ con `solve_triangular`. Por eso el log-determinante lleva signo negativo.
- `Chol3`: igual que `Chol1`, pero invierte con `dtrtri`, una rutina hecha justo para matrices triangulares.

### Puntos 12 y 13. `TensorizedChol` y `EfficientChol`
Heredan de `QDA_Chol3` (en el benchmark `Chol1` y `Chol3` rindieron igual por lo que se eligió `Chol3`). La inversa y el log-determinante se calculan una sola vez, en `fit`, y la predicción es solo un producto de matrices, que se puede apilar por clases. (`Chol2` no sirve para esto porque `solve_triangular` no trabaja en lote.)

`TensorizedChol` paraleliza sobre las clases. `EfficientChol` paraleliza además sobre las observaciones, y no aparece ninguna matriz de $n \times n$: elevar al cuadrado y sumar por columnas ya equivale a quedarse con la diagonal.

In [33]:
class TensorizedChol(QDA_Chol3):
    """Paraleliza sobre las k clases (una observacion por vez)."""

    def _fit_params(self, X, y):
        super()._fit_params(X, y)
        self.tensor_L_invs = np.stack(self.L_invs)     # (k, p, p)
        self.tensor_means = np.stack(self.means)       # (k, p, 1)
        # log |Sigma|^(-1/2) = suma de log(diag(L^-1))
        self.log_dets = np.log(np.diagonal(self.tensor_L_invs, axis1=1, axis2=2)).sum(axis=1)  # (k,)

    def _predict_log_conditionals(self, x):
        z = self.tensor_L_invs @ (x - self.tensor_means)          # (k, p, 1)
        return self.log_dets - 0.5 * (z ** 2).sum(axis=(1, 2))    # (k,)

    def _predict_one(self, x):
        return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))


class EfficientChol(TensorizedChol):
    """Paraleliza sobre clases y observaciones. No aparece ninguna matriz n x n."""

    def predict(self, X):
        unbiased_x = X[np.newaxis, :, :] - self.tensor_means      # (k, p, n)
        z = self.tensor_L_invs @ unbiased_x                       # (k, p, n)
        quad = np.sum(z ** 2, axis=1)                             # (k, n)
        scores = self.log_a_priori[:, None] + self.log_dets[:, None] - 0.5 * quad
        return np.argmax(scores, axis=0).reshape(1, -1)

### Verificación de las 9 variantes

In [34]:
ALL_MODELS = [QDA, TensorizedQDA, FasterQDA, EfficientQDA,
              QDA_Chol1, QDA_Chol2, QDA_Chol3, TensorizedChol, EfficientChol]
preds = {}
for M in ALL_MODELS:
    m = M(); m.fit(Xtr, ytr); preds[M.__name__] = m.predict(Xte)
print({name: bool((p_.ravel() == pred_ref).all()) for name, p_ in preds.items()})

# comparación de los log-condicionales (no solo de las clases predichas)
ef = EfficientQDA(); ef.fit(Xtr, ytr)
print("EfficientQDA, log-condicionales:", np.allclose(ef._predict_log_conditionals_batch(Xte), ref_cond))
fa = FasterQDA(); fa.fit(Xtr, ytr)
print("FasterQDA, log-condicionales:   ", np.allclose(fa._predict_log_conditionals_batch(Xte), ref_cond))
ec = EfficientChol(); ec.fit(Xtr, ytr)
z = ec.tensor_L_invs @ (Xte[None] - ec.tensor_means)
print("EfficientChol, log-condicionales:", np.allclose(ec.log_dets[:, None] - 0.5 * (z**2).sum(1), ref_cond))

{'QDA': True, 'TensorizedQDA': True, 'FasterQDA': True, 'EfficientQDA': True, 'QDA_Chol1': True, 'QDA_Chol2': True, 'QDA_Chol3': True, 'TensorizedChol': True, 'EfficientChol': True}
EfficientQDA, log-condicionales: True
FasterQDA, log-condicionales:    True
EfficientChol, log-condicionales: True


## 7. Benchmark

- `DATASET = "letters"` es el del ejemplo proporcionado (20000 observaciones, 16 variables, 26 clases).
- `FULL_RUNS = False` usa pocas repeticiones (rápido). `FULL_RUNS = True` usa las del cuaderno prporcionado (100 corridas) y logicamente tarda mucho mas , pero aun asi utilizo `True`.
- `same_splits=True`: todos los modelos se prueban con los mismos datos, así sus precisiones son comparables.

**Memoria:** `FasterQDA` con Letters usa unos 3 GB. La sesión gratuita de Colab (que provee unos 12 GB) lo tolera pero se puede bajar `TEST_SZ` (por ejemplo a 0.1).

In [35]:
DATASET = "letters"     # "letters" | "wine" | "synthetic"
FULL_RUNS = True       # lo cambie a True
TEST_SZ = 0.2

def make_synthetic(n=6000, p=16, k=26, seed=0):
    rng = RandomState(seed)
    mus = rng.randn(k, p) * 2
    mats = [rng.randn(p, p) * 0.5 + np.eye(p) for _ in range(k)]
    y = rng.randint(0, k, n)
    X = np.stack([mus[c] + mats[c] @ rng.randn(p) for c in y])
    return X, y.reshape(-1, 1)

def load_bench_data(name):
    if name == "wine":
        X, y = get_wine_dataset(); return X, label_encode(y), "wine"
    if name == "letters":
        try:
            X, y = get_letters_dataset(); return X, label_encode(y), "letters"
        except Exception as e:
            print("No se pudo descargar Letters (" + type(e).__name__ + "). Se usa un dataset sintetico de la misma forma.")
    X, y = make_synthetic(); return X, y, "synthetic (n=6000, p=16, k=26)"

X_b, y_b, used = load_bench_data(DATASET)
print("Dataset usado:", used, "| X:", X_b.shape, "| clases:", len(np.unique(y_b)))

runs = dict(n_runs=100, warmup=20, mem_runs=30) if FULL_RUNS else dict(n_runs=10, warmup=2, mem_runs=3)
b = Benchmark(X_b, y_b, test_sz=TEST_SZ, same_splits=True, **runs)

Dataset usado: letters | X: (20000, 16) | clases: 26
Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2


### Punto 7. Comparación de las 4 variantes sin Cholesky

In [36]:
for M in [QDA, TensorizedQDA, FasterQDA, EfficientQDA]:
    b.bench(M)
cols = ['train_median_ms', 'test_median_ms', 'mean_accuracy', 'test_mem_median_mb', 'test_speedup']
b.summary(baseline='QDA')[cols].round(3)

QDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

FasterQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

FasterQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

EfficientQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

,train_median_ms,test_median_ms,mean_accuracy,test_mem_median_mb,test_speedup
model,,,,,
QDA,16.220,2597.381,0.886,0.098,1.000
TensorizedQDA,16.195,472.914,0.886,0.154,5.492
FasterQDA,22.676,1762.611,0.886,3199.335,1.474
EfficientQDA,22.163,36.397,0.886,38.202,71.362


### Punto 11. Comparación de las 7 variantes (se agregan las 3 de Cholesky)

In [37]:
for M in [QDA_Chol1, QDA_Chol2, QDA_Chol3]:
    b.bench(M)
cols = ['train_median_ms', 'test_median_ms', 'mean_accuracy', 'test_mem_median_mb', 'test_speedup']
b.summary(baseline='QDA')[cols].round(3)

QDA_Chol1 (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA_Chol1 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol2 (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA_Chol2 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol3 (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA_Chol3 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

,train_median_ms,test_median_ms,mean_accuracy,test_mem_median_mb,test_speedup
model,,,,,
QDA,16.220,2597.381,0.886,0.098,1.000
TensorizedQDA,16.195,472.914,0.886,0.154,5.492
FasterQDA,22.676,1762.611,0.886,3199.335,1.474
EfficientQDA,22.163,36.397,0.886,38.202,71.362
QDA_Chol1,17.928,1406.733,0.886,0.095,1.846
QDA_Chol2,17.179,4630.783,0.886,0.096,0.561
QDA_Chol3,16.458,1290.348,0.886,0.095,2.013


### Punto 14. Comparación de las 9 variantes (se agregan `TensorizedChol` y `EfficientChol`)

In [38]:
for M in [TensorizedChol, EfficientChol]:
    b.bench(M)
summ = b.summary(baseline='QDA')
summ[['train_median_ms', 'train_speedup', 'test_median_ms', 'test_speedup',
      'mean_accuracy', 'train_mem_median_mb', 'test_mem_median_mb']].round(3)

TensorizedChol (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientChol (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

EfficientChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

,train_median_ms,train_speedup,test_median_ms,test_speedup,mean_accuracy,train_mem_median_mb,test_mem_median_mb
model,,,,,,,
QDA,16.220,1.000,2597.381,1.000,0.886,0.269,0.098
TensorizedQDA,16.195,1.002,472.914,5.492,0.886,0.269,0.154
FasterQDA,22.676,0.715,1762.611,1.474,0.886,0.269,3199.335
EfficientQDA,22.163,0.732,36.397,71.362,0.886,0.269,38.202
QDA_Chol1,17.928,0.905,1406.733,1.846,0.886,0.269,0.095
QDA_Chol2,17.179,0.944,4630.783,0.561,0.886,0.269,0.096
QDA_Chol3,16.458,0.986,1290.348,2.013,0.886,0.269,0.095
TensorizedChol,17.068,0.950,84.264,30.825,0.886,0.269,0.158
EfficientChol,21.269,0.763,28.832,90.085,0.886,0.269,39.059


## 8. Versiones utilizadas

In [39]:
print("Python", sys.version.split()[0])
print("NumPy", np.__version__)
print("SciPy", scipy.__version__)
print("Dataset del benchmark:", used)

Python 3.13.16
NumPy 2.1.3
SciPy 1.16.3
Dataset del benchmark: letters


## 9. Conclusiones

Los números provienen de la ejecución con **Letters** (16000 observaciones de entrenamiento, 4000 de prueba, 16 variables, 26 clases), con las repeticiones del enunciado, es decir 100 de tiempo, 20 de calentamiento y 30 de memoria (versiones en la sección 8). En todas las tablas se informa la mediana.

**Todos los modelos aciertan igual (0.886).** Esto confirma que las 9 variantes hacen la misma cuenta: solo cambia la velocidad.

**Punto 7. Las 4 variantes sin Cholesky**
- `QDA` original: 2597 ms para predecir las 4000 observaciones.
- `TensorizedQDA`: 473 ms (unas 5.5 veces más rápido). Ya no recorre las clases de a una.
- `FasterQDA`: 1763 ms (solo 1.5 veces más rápido) y usa unos 3.2 GB de memoria, contra 0.1 MB. Mejora a `QDA`, pero la matriz $n \times n$ le quita casi toda la ventaja y queda peor que `TensorizedQDA`.
- `EfficientQDA`: 36 ms (unas 71 veces más rápido) con solo 38 MB.
- ¿Se condice con lo esperado? Sí. Procesar todo junto ayuda, pero armar la matriz $n \times n$ no es un una buena decision por lo que evitarla es lo que da la gran mejora.

**Punto 11. Las 7 variantes**
- `Chol1` y `Chol3` rinden casi igual (1407 y 1290 ms, unas 2 veces mejor que `QDA`). Su código de predicción es idéntico, así que la pequeña diferencia es de medición. Con 16 variables da lo mismo usar la inversa general o la específica para triangulares.
- `Chol2` es la peor: 4631 ms, casi el doble de lenta que `QDA`. Llama a `solve_triangular` una vez por observación y por clase (unas 100 000 llamadas), y cada llamada tiene un costo fijo que se acumula.
- No hay una `Chol` claramente mejor: `Chol1` y `Chol3` empatan, y ambas le ganan a `Chol2`.
- Los tiempos de entrenamiento son parecidos en todos (16 a 23 ms). Con matrices tan chicas, Cholesky no acelera el ajuste. `TensorizedQDA`, `FasterQDA` y `EfficientQDA` usan exactamente el mismo `fit` y aun así dan 16, 23 y 22 ms. Son variaciones de la medición.

**Punto 14. Las 9 variantes**
- `EfficientChol` es la mejor con 29 ms, unas 90 veces más rápida que `QDA`, con 39 MB.
- `TensorizedChol` (84 ms) es unas 5.6 veces más rápida que `TensorizedQDA` (473 ms). `TensorizedQDA` recalcula el determinante en cada observación, mientras que `TensorizedChol` lo calcula una sola vez en `fit`.
- `EfficientChol` es solo un poco más rápida que `EfficientQDA` (29 ms contra 36 ms, unas 1.3 veces), con la misma memoria. Con pocas repeticiones esa ventaja parecía mayor, pero con 100 corridas queda modesta.
- ¿Se condice con lo esperado? En parte. Se esperaba que Cholesky ayudara y ayuda, pero con 16 variables las matrices son tan chicas que la ganancia sobre la versión sin Cholesky es pequeña en las variantes eficientes. Donde más se nota es al comparar `TensorizedChol` con `TensorizedQDA`.

**En resumen:** lo que más pesaba era el `for` de Python. Después, evitar la matriz $n \times n$. Cholesky da una mejora adicional, mayor en la versión por clases y modesta en la versión eficiente. Al menos eso es lo que observo. Habria que profundizar aun mas con mas experimentos.

In [40]:
summ[['test_median_ms', 'test_speedup', 'test_mem_median_mb', 'mean_accuracy']].sort_values('test_median_ms').round(3)

,test_median_ms,test_speedup,test_mem_median_mb,mean_accuracy
model,,,,
EfficientChol,28.832,90.085,39.059,0.886
EfficientQDA,36.397,71.362,38.202,0.886
TensorizedChol,84.264,30.825,0.158,0.886
TensorizedQDA,472.914,5.492,0.154,0.886
QDA_Chol3,1290.348,2.013,0.095,0.886
QDA_Chol1,1406.733,1.846,0.095,0.886
FasterQDA,1762.611,1.474,3199.335,0.886
QDA,2597.381,1.000,0.098,0.886
QDA_Chol2,4630.783,0.561,0.096,0.886


## Nota:
 El ejemplo del provisto usa `same_splits=False`, así que cada modelo se evalúa con datos distintos. En este caso use `True` para comparar precisiones de forma pareja, pero estaria interesante analizar tambien usando False.
